In [1]:
import os
import re
import zlib
import fitz          # pymupdf: PDF 파싱
import olefile       # HWP 파싱
import pandas as pd
from pathlib import Path
from collections import Counter

In [2]:
import time
import json
import uuid
import shutil
import tempfile
import subprocess
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict
from bs4 import BeautifulSoup
import unicodedata

## (0)흐름
- pdf 재파싱 및 노이즈 제거<br>
- hwp 재파싱 및 노이즈 제거<br>
- pdf,hwp concat

## (1)데이터 불러오기

In [4]:
folder = Path("/home/shared/files") 

ext_counter = Counter()

for f in folder.rglob("*"):
    if f.is_file():
        ext = f.suffix.lower() if f.suffix else "(no_extension)"
        ext_counter[ext] += 1

print("확장자별 개수")
for ext, count in sorted(ext_counter.items()):
    print(f"{ext}: {count}")

확장자별 개수
.docx: 1
.hwp: 94
.pdf: 6


## (2)pdf 파싱/정제

In [5]:
pdf_folder = Path("/home/shared/files") 
pdf_files = sorted([path for path in pdf_folder.glob("*.pdf") if path.is_file()])


# PDF 파일 1개에서 페이지별 텍스트를 추출해 raw_text를 만들 때 사용
def extract_pdf_text(pdf_path: Path) -> str:
    page_texts = []

    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text") or ""
            page_texts.append(text)

    return "\n".join(page_texts).strip()


# 띄어져 깨진 단어(예: 제 안 요 청 서)도 찾기 위한 정규식 패턴을 만들 때 사용
def build_spaced_phrase_pattern(phrase: str) -> str:
    chars = [re.escape(ch) for ch in phrase if ch.strip()]
    return r"(?<![가-힣A-Za-z0-9])" + r"\s*".join(chars) + r"(?![가-힣A-Za-z0-9])"


# 문서 내 핵심 용어의 비정상 띄어쓰기를 복원해 검색/청킹 품질을 높일 때 사용
def fix_spaced_phrases(text: str, phrases=None) -> str:
    if phrases is None:
        phrases = [
            "제안요청서",
            "사업명",
            "사업비",
            "사업기간",
            "사업범위",
            "사업개요",
            "주관기관",
            "제안안내",
            "제안서",
            "제안요청",
            "통합시스템",
            "입학처",
            "서울시립대학교",
            "학업성취도",
            "종단분석",
            "추진배경",
            "필요성",
            "요구사항",
            "상세설명",
            "세부내용",
            "유지보수",
            "보안요구사항",
            "성능요구사항",
            "인터페이스",
            "데이터베이스",
            "사용자",
            "관리자",
            "클라우드",
            "인프라",
            "프로파일링",
        ]

    for phrase in phrases:
        text = re.sub(build_spaced_phrase_pattern(phrase), phrase, text)

    return text


# 목차/페이지번호/반복 헤더 같은 전형적인 PDF 노이즈 라인을 제거할 때 사용
def strip_toc_and_noise_lines(text: str) -> str:
    cleaned_lines = []
    in_front_toc = False
    pending_toc_title = False

    for line in text.split("\n"):
        stripped = line.strip()
        compact = re.sub(r"\s+", "", stripped)

        if compact == "목차":
            in_front_toc = True
            pending_toc_title = False
            continue

        if stripped == "목":
            pending_toc_title = True
            continue

        if pending_toc_title and stripped == "차":
            in_front_toc = True
            pending_toc_title = False
            continue

        pending_toc_title = False

        if in_front_toc:
            toc_like = (
                re.search(r"[\.·]{4,}\s*\d{1,3}$", stripped) is not None
                or re.match(r"^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+\s*[\.)]?\s*[가-힣A-Za-z]", stripped) is not None
                or re.match(r"^\d+\s*[\.)]\s*[가-힣A-Za-z]", stripped) is not None
                or compact in {"l", "I"}
            )

            if not stripped or toc_like:
                continue

            if re.fullmatch(r"-\s*\d{1,3}\s*-", stripped):
                in_front_toc = False
                continue

            in_front_toc = False

        if not stripped:
            cleaned_lines.append("")
            continue

        if compact in {"[사전공개용]", "l", "I"}:
            continue

        if re.fullmatch(r"\d{1,3}", compact):
            continue

        if re.fullmatch(r"-\s*\d{1,3}\s*-", stripped):
            continue

        if re.fullmatch(r"페\s*이\s*지\s*:?\s*\d+\s*/\s*\d+", stripped):
            continue

        if re.fullmatch(r"(문\s*서\s*번\s*호|개\s*정\s*번\s*호|발\s*행\s*일)\s*:?", stripped):
            continue

        if re.search(r"[\.·]{4,}\s*\d{1,3}$", stripped):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


# 자주 발생하는 깨짐 표현을 사전 치환 방식으로 후처리할 때 사용
def fix_additional_phrases(text: str) -> str:
    replacements = {
        "기개 발": "기개발",
        "시 스템": "시스템",
        "데 이터": "데이터",
        "클 라우드": "클라우드",
        "인 프라": "인프라",
        "상 세 설명": "상세설명",
        "세 부 내용": "세부내용",
        "사 업 비": "사업비",
        "사 업 명": "사업명",
        "사 업 기 간": "사업기간",
        "사 업 범 위": "사업범위",
        "주 관 기 관": "주관기관",
        "제 안 서": "제안서",
        "제 안 안 내": "제안안내",
        "요 구 사 항": "요구사항",
        "성 능 요 구 사 항": "성능요구사항",
        "보 안 요 구 사 항": "보안요구사항",
        "데 이 터 베 이 스": "데이터베이스",
        "프 로 파 일 링": "프로파일링",
        "용 역 업 체": "용역업체",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


# 붙임/별첨/서식/입찰안내 등 본문 이후 부록성 구간을 잘라 RAG 노이즈를 줄일 때 사용
def truncate_appendix_sections(text: str) -> str:
    cutoff_patterns = [
        r"\[\s*붙임\s*\d+\s*\]",
        r"\[\s*별첨\s*\d*\s*\]",
        r"\[\s*별지서식\s*\d*\s*\]",
        r"\[\s*서식\s*\d+\s*\]",
        r"별첨\s*[IVX]+\s*제안서\s*관련\s*서식",
        r"제안서\s*관련\s*서식",
        r"서식\s*\d+\s*:",
        r"제\s*[IVX]+\s*장\s*제안서\s*제출안내",
        r"제\s*[IVX]+\s*장\s*제안서\s*작성기준",
        r"제\s*[IVX]+\s*장\s*입찰\s*안내",
        r"입찰참가신청서",
        r"입찰\s*참가등록",
        r"입찰\s*참가\s*자격",
        r"입찰서류\s*및\s*제안서\s*제출",
        r"제안서\s*제출안내",
        r"제출서류",
        r"제안서\s*작성기준",
        r"제안서\s*작성방법",
        r"제안서\s*작성지침",
        r"제안서\s*작성지침\s*및\s*유의사항",
        r"입찰안내\s*사항",
        r"제안서\s*평가\s*및\s*협상",
        r"제안사\s*유의사항",
        r"입찰\s*공고문",
        r"입찰에\s*참가하고자\s*하는\s*자",
        r"청렴계약\s*서약서",
        r"안전보건관리\s*준수서약서",
        r"정량평가(?:기준|지표)",
        r"제안요구사항\s*수용\s*조견표",
        r"하도급계약\s*적정성",
        r"자가진단표",
    ]

    cutoff_idx = None
    min_start = int(len(text) * 0.35)

    for pattern in cutoff_patterns:
        for match in re.finditer(pattern, text):
            if match.start() >= min_start:
                cutoff_idx = match.start() if cutoff_idx is None else min(cutoff_idx, match.start())
                break

    if cutoff_idx is not None:
        text = text[:cutoff_idx]

    return text


# 서명란 같은 메타 정보를 지워 RAG 검색 잡음을 줄일 때 사용
def remove_contact_and_signature_noise(text: str) -> str:
    text = re.sub(r"\(인\)|서명|날인", " ", text)
    return text


# raw_text 전체에 정규화/노이즈 제거/표현 복원을 순차 적용해 clean_text를 생성할 때 사용
def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\t\f\v]+", " ", text)
    text = re.sub(r"[  ]{2,}", " ", text)
    text = re.sub(r"[^가-힣A-Za-z0-9\s\.\(\)\[\]\/,%:\-·?!@~&+*'\"=<>_]", " ", text)
    text = re.sub(r"페\s*이\s*지\s*:\s*\n?\s*\d+\s*/\s*\d+", " ", text)
    text = re.sub(r"(문\s*서\s*번\s*호|개\s*정\s*번\s*호|발\s*행\s*일)\s*:\s*\n?\s*[-\d\. ]+", " ", text)
    text = strip_toc_and_noise_lines(text)
    text = re.sub(r"(?ms)^(?:[IVX]+\.\s*[^\n]+\n\s*){2,8}", "", text, count=1)
    text = re.sub(r"(?m)^\s*[IVX]+\.\s*(?:제안|프로젝트|자격|수행능력|계약|기타|현황|개요|부문)[^\n]*$", "", text)
    text = re.sub(r"(?m)^[ \t]*[-−–]\s*\d{1,4}\s*[-−–][ \t]*$", " ", text)
    text = re.sub(r"(?<!\d)[-−–]\s*\d{1,4}\s*[-−–](?!\d)", " ", text)
    text = re.sub(r"[\.·]{4,}", " ", text)
    text = re.sub(r"[-_=]{3,}", " ", text)
    text = fix_spaced_phrases(text)
    text = fix_additional_phrases(text)
    text = truncate_appendix_sections(text)
    text = remove_contact_and_signature_noise(text)
    text = re.sub(r"\n[ ]+", "\n", text)
    text = re.sub(r"[ ]+\n", "\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    text = re.sub(r"(?<![\.!?])\n(?!\n)", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


parsed_rows = []

for pdf_path in pdf_files:
    raw_text = extract_pdf_text(pdf_path)
    clean_text = clean_pdf_text(raw_text)

    parsed_rows.append(
        {
            "파일명": pdf_path.name,
            "파일형식": "pdf",
            "raw_text": raw_text,
            "clean_text": clean_text,
        }
    )

df_parsed_pdf = pd.DataFrame(
    parsed_rows,
    columns=["파일명", "파일형식", "raw_text", "clean_text"],
)

print(f"파싱 대상 PDF 수: {len(pdf_files)}")
display(df_parsed_pdf)

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

파싱 대상 PDF 수: 6


,파일명,파일형식,raw_text,clean_text
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,pdf,-1- \n \n \n \n제 안 요 청 서 \n \n \n고려대학교 \n차세대 ...,제안요청서\n 고려대학교 차세대 포털·학사 정보시스템 구축 사업\n 2024. 7....
1,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,pdf,2025년도 중이온가속기용 극저온시스템\n운전 용역 과업지시서\n2024. 10.\...,2025년도 중이온가속기용 극저온시스템 운전 용역 과업지시서 2024. 10.\n중...
2,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,pdf,(재공고)대전대학교 다층적 융합 학습경험\n플랫폼(MILE) 구축 제안요청서 \n2...,(재공고)대전대학교 다층적 융합 학습경험 플랫폼(MILE) 구축 제안요청서 2024...
3,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,pdf,[사전공개용]\n제 안 요 청 서\n본 제안요청서는 입찰참여의 균등한 기회 제공을 ...,제안요청서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한...
4,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,pdf,제 안 요 청 서\n사 업 명\n2024년 지도정보 플랫폼 및 전문활용 \n연계 시...,제안요청서 사업명 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역 ...
5,한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp...,pdf,『아세안+3 식량안보정보시스템 3단계 협력사업(캄보디아)』\nPMC 용역 제안요청서...,아세안+3 식량안보정보시스템 3단계 협력사업(캄보디아) PMC 용역 제안요청서 20...


In [6]:
df_text_len_check = df_parsed_pdf.copy()

df_text_len_check["raw_len"] = df_text_len_check["raw_text"].fillna("").str.len()
df_text_len_check["clean_len"] = df_text_len_check["clean_text"].fillna("").str.len()
df_text_len_check["length_diff"] = df_text_len_check["raw_len"] - df_text_len_check["clean_len"]
df_text_len_check["clean_ratio(%)"] = (df_text_len_check["clean_len"] / df_text_len_check["raw_len"] * 100).round(2)

display(
    df_text_len_check[
        ["파일명", "raw_len", "clean_len", "length_diff", "clean_ratio(%)"]
    ].sort_values("length_diff", ascending=False)
)

print("평균 clean_ratio(%):", df_text_len_check["clean_ratio(%)"].mean().round(2))

,파일명,raw_len,clean_len,length_diff,clean_ratio(%)
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,232583,71571,161012,30.77
4,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,119969,39244,80725,32.71
3,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,116371,36739,79632,31.57
5,한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp...,101627,36203,65424,35.62
2,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,67574,27394,40180,40.54
1,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,45609,20184,25425,44.25


평균 clean_ratio(%): 35.91


In [7]:
from pathlib import Path

output_dir = Path("/home/bidcoin/tmp") 
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / "df_parsed_pdf.csv"
df_parsed_pdf[["파일명", "파일형식", "clean_text"]].to_csv(output_csv_path, index=False, encoding="utf-8-sig")

print(f"CSV 저장 완료: {output_csv_path}")
print(f"저장 행 수: {len(df_parsed_pdf)}")

CSV 저장 완료: /home/bidcoin/tmp/df_parsed_pdf.csv
저장 행 수: 6


In [8]:
def make_rag_text(text: str) -> str:
    text = text or ""

    hard_cut_patterns = [
        r"\[\s*붙임\s*\d+\s*\]",
        r"\[\s*별첨\s*\d*\s*\]",
        r"\[\s*별지서식\s*\d*\s*\]",
        r"\[\s*서식\s*\d+\s*\]",
        r"제\s*[IVX]+\s*장\s*제안서\s*제출안내",
        r"제\s*[IVX]+\s*장\s*제안서\s*작성기준",
        r"제\s*[IVX]+\s*장\s*입찰\s*안내",
        r"입찰\s*참가등록",
        r"입찰\s*참가\s*자격",
        r"입찰서류\s*및\s*제안서\s*제출",
        r"제안서\s*제출안내",
        r"제출서류",
        r"제안서\s*작성기준",
        r"제안서\s*작성방법",
        r"제안서\s*작성지침",
        r"제안서\s*평가\s*및\s*협상",
        r"제안사\s*유의사항",
        r"입찰안내\s*사항",
        r"입찰\s*공고문",
        r"입찰에\s*참가하고자\s*하는\s*자",
        r"정량평가(?:기준|지표)",
        r"제안요구사항\s*수용\s*조견표",
        r"하도급계약\s*적정성",
        r"자가진단표",
    ]

    cutoff_idx = None
    min_start = int(len(text) * 0.25)

    for pattern in hard_cut_patterns:
        for match in re.finditer(pattern, text):
            if match.start() >= min_start:
                cutoff_idx = match.start() if cutoff_idx is None else min(cutoff_idx, match.start())
                break

    if cutoff_idx is not None:
        text = text[:cutoff_idx]

    text = re.sub(r"\(인\)|서명|날인", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


df_parsed_pdf_hard = df_parsed_pdf[["파일명", "파일형식", "raw_text", "clean_text"]].copy()
df_parsed_pdf_hard["rag_text"] = df_parsed_pdf_hard["clean_text"].apply(make_rag_text)

display(df_parsed_pdf_hard.head())

,파일명,파일형식,raw_text,clean_text,rag_text
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,pdf,-1- \n \n \n \n제 안 요 청 서 \n \n \n고려대학교 \n차세대 ...,제안요청서\n 고려대학교 차세대 포털·학사 정보시스템 구축 사업\n 2024. 7....,제안요청서\n 고려대학교 차세대 포털·학사 정보시스템 구축 사업\n 2024. 7....
1,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,pdf,2025년도 중이온가속기용 극저온시스템\n운전 용역 과업지시서\n2024. 10.\...,2025년도 중이온가속기용 극저온시스템 운전 용역 과업지시서 2024. 10.\n중...,2025년도 중이온가속기용 극저온시스템 운전 용역 과업지시서 2024. 10.\n중...
2,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,pdf,(재공고)대전대학교 다층적 융합 학습경험\n플랫폼(MILE) 구축 제안요청서 \n2...,(재공고)대전대학교 다층적 융합 학습경험 플랫폼(MILE) 구축 제안요청서 2024...,(재공고)대전대학교 다층적 융합 학습경험 플랫폼(MILE) 구축 제안요청서 2024...
3,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,pdf,[사전공개용]\n제 안 요 청 서\n본 제안요청서는 입찰참여의 균등한 기회 제공을 ...,제안요청서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한...,제안요청서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한...
4,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,pdf,제 안 요 청 서\n사 업 명\n2024년 지도정보 플랫폼 및 전문활용 \n연계 시...,제안요청서 사업명 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역 ...,제안요청서 사업명 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역 ...


### (2-1)저장

In [9]:
# 저장
save_dir = Path("/home/bidcoin")
save_dir.mkdir(parents=True, exist_ok=True)

save_hard_csv_path = save_dir / "df_parsed_pdf_hard.csv"
df_parsed_pdf_hard.to_csv(save_hard_csv_path, index=False, encoding="utf-8")

print(f"Hard CSV 저장 완료: {save_hard_csv_path}")
print(f"저장 행 수: {len(df_parsed_pdf_hard)}")

Hard CSV 저장 완료: /home/bidcoin/df_parsed_pdf_hard.csv
저장 행 수: 6


### (2-2)확인용

In [10]:
output_dir

PosixPath('/home/bidcoin/tmp')

In [11]:
rag_text_dir = output_dir / "rag_text_txt"
rag_text_dir.mkdir(parents=True, exist_ok=True)

for row in df_parsed_pdf_hard.itertuples(index=False):
    txt_name = Path(row.파일명).stem + ".txt"
    txt_path = rag_text_dir / txt_name
    txt_path.write_text(row.rag_text, encoding="utf-8")

print(f"rag_text txt 저장 폴더: {rag_text_dir}")
print(f"저장 파일 수: {len(df_parsed_pdf_hard)}")

rag_text txt 저장 폴더: /home/bidcoin/tmp/rag_text_txt
저장 파일 수: 6


In [12]:
clean_text_dir = output_dir / "clean_text_txt"
clean_text_dir.mkdir(parents=True, exist_ok=True)

for row in df_parsed_pdf.itertuples(index=False):
    txt_name = Path(row.파일명).stem + ".txt"
    txt_path = clean_text_dir / txt_name
    txt_path.write_text(row.clean_text, encoding="utf-8")

print(f"clean_text txt 저장 폴더: {clean_text_dir}")
print(f"저장 파일 수: {len(df_parsed_pdf)}")

clean_text txt 저장 폴더: /home/bidcoin/tmp/clean_text_txt
저장 파일 수: 6


In [13]:
raw_text_dir = output_dir / "raw_text_txt"
raw_text_dir.mkdir(parents=True, exist_ok=True)

for row in df_parsed_pdf.itertuples(index=False):
    txt_name = Path(row.파일명).stem + ".txt"
    txt_path = raw_text_dir / txt_name
    txt_path.write_text(row.raw_text, encoding="utf-8")

print(f"raw_text txt 저장 폴더: {raw_text_dir}")
print(f"저장 파일 수: {len(df_parsed_pdf)}")

raw_text txt 저장 폴더: /home/bidcoin/tmp/raw_text_txt
저장 파일 수: 6


In [14]:
df_text_compare = df_parsed_pdf[["파일명", "파일형식", "raw_text", "clean_text"]].copy()
df_text_compare = df_text_compare.merge(
    df_parsed_pdf_hard[["파일명", "rag_text"]],
    on="파일명",
    how="left",
)

df_text_compare["raw_len"] = df_text_compare["raw_text"].fillna("").str.len()
df_text_compare["clean_len"] = df_text_compare["clean_text"].fillna("").str.len()
df_text_compare["rag_len"] = df_text_compare["rag_text"].fillna("").str.len()

df_text_compare["raw_to_clean_diff"] = df_text_compare["raw_len"] - df_text_compare["clean_len"]
df_text_compare["clean_to_rag_diff"] = df_text_compare["clean_len"] - df_text_compare["rag_len"]
df_text_compare["raw_to_rag_diff"] = df_text_compare["raw_len"] - df_text_compare["rag_len"]

df_text_compare["clean_ratio(%)"] = (df_text_compare["clean_len"] / df_text_compare["raw_len"] * 100).round(2)
df_text_compare["rag_ratio_vs_raw(%)"] = (df_text_compare["rag_len"] / df_text_compare["raw_len"] * 100).round(2)
df_text_compare["rag_ratio_vs_clean(%)"] = (df_text_compare["rag_len"] / df_text_compare["clean_len"] * 100).round(2)

display(
    df_text_compare[
        [
            "파일명",
            "raw_len",
            "clean_len",
            "rag_len",
            "raw_to_clean_diff",
            "clean_to_rag_diff",
            "raw_to_rag_diff",
            "clean_ratio(%)",
            "rag_ratio_vs_raw(%)",
            "rag_ratio_vs_clean(%)",
        ]
    ].sort_values("raw_to_rag_diff", ascending=False)
)

print("평균 clean_ratio(%):", df_text_compare["clean_ratio(%)"].mean().round(2))
print("평균 rag_ratio_vs_raw(%):", df_text_compare["rag_ratio_vs_raw(%)"].mean().round(2))
print("평균 rag_ratio_vs_clean(%):", df_text_compare["rag_ratio_vs_clean(%)"].mean().round(2))

,파일명,raw_len,clean_len,rag_len,raw_to_clean_diff,clean_to_rag_diff,raw_to_rag_diff,clean_ratio(%),rag_ratio_vs_raw(%),rag_ratio_vs_clean(%)
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,232583,71571,47478,161012,24093,185105,30.77,20.41,66.34
3,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,116371,36739,24101,79632,12638,92270,31.57,20.71,65.60
4,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,119969,39244,39244,80725,0,80725,32.71,32.71,100.00
5,한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp...,101627,36203,36203,65424,0,65424,35.62,35.62,100.00
2,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,67574,27394,16624,40180,10770,50950,40.54,24.60,60.68
1,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,45609,20184,20184,25425,0,25425,44.25,44.25,100.00


평균 clean_ratio(%): 35.91
평균 rag_ratio_vs_raw(%): 29.72
평균 rag_ratio_vs_clean(%): 82.1


In [15]:
# 최종파일 : df_parsed_pdf_hard
df_parsed_pdf_hard.columns

Index(['파일명', '파일형식', 'raw_text', 'clean_text', 'rag_text'], dtype='str')

***

## (3)hwp 파싱

### (3-1)파싱함수

In [16]:
def parse_hwp(file_path: str) -> str:
    """
    HWP 파일에서 raw_text만 추출
    - 구조추출/표추출 없음
    - rec_type == 67 텍스트 레코드만 사용
    """
    section_texts = []

    try:
        with olefile.OleFileIO(file_path) as ole:
            if not ole.exists("BodyText"):
                return ""

            section_idx = 0
            while ole.exists(f"BodyText/Section{section_idx}"):
                data = ole.openstream(f"BodyText/Section{section_idx}").read()

                # 압축 해제 시도
                try:
                    data = zlib.decompress(data, -15)
                except zlib.error:
                    # 압축 안 된 경우 그대로 사용
                    pass

                chars = []
                i = 0

                while i < len(data):
                    # 헤더 4바이트 못 읽으면 종료
                    if i + 4 > len(data):
                        break

                    header = int.from_bytes(data[i:i+4], "little")
                    rec_type = header & 0x3FF
                    rec_len = (header >> 20) & 0xFFF
                    i += 4

                    # 확장 길이 처리
                    if rec_len == 0xFFF:
                        if i + 4 > len(data):
                            break
                        rec_len = int.from_bytes(data[i:i+4], "little")
                        i += 4

                    # body 범위 체크
                    if i + rec_len > len(data):
                        break

                    body = data[i:i+rec_len]
                    i += rec_len

                    # 본문 텍스트 레코드만 추출
                    if rec_type == 67:
                        for j in range(0, len(body) - 1, 2):
                            ch = int.from_bytes(body[j:j+2], "little")

                            if ch in (0x0D, 0x0A, 10, 13, 0):
                                chars.append("\n")
                            elif 0x20 <= ch <= 0xD7A3 or ch > 0xE000:
                                try:
                                    chars.append(chr(ch))
                                except ValueError:
                                    pass

                section_text = "".join(chars).strip()
                if section_text:
                    section_texts.append(section_text)

                section_idx += 1

    except Exception:
        return ""

    return "\n".join(section_texts).strip()

### (3-2)파일1개를 raw_only로 파싱

In [17]:
def parse_single_hwp_raw(file_path: str) -> dict:
    """
    HWP 파일 1개를 raw-only 방식으로 파싱해서 dict 반환
    """
    start_time = time.time()

    file_name = os.path.basename(file_path)
    file_ext = Path(file_path).suffix.lower().replace(".", "")

    raw_text = ""
    status = "success"
    parse_warning = None

    try:
        raw_text = parse_hwp(file_path)

        if not raw_text:
            status = "empty"
            parse_warning = "raw_text is empty"

    except Exception as e:
        raw_text = ""
        status = "error"
        parse_warning = str(e)

    processing_time = round(time.time() - start_time, 2)
    raw_text_len = len(raw_text) if raw_text else 0
    raw_parse_success = raw_text_len > 0

    return {
        "파일명": file_name,
        "파일형식": file_ext,
        "파일경로": str(file_path),
        "raw_text": raw_text,
        "raw_text_len": raw_text_len,
        "raw_parse_success": raw_parse_success,
        "status": status,
        "parse_warning": parse_warning,
        "processing_time": processing_time,
        "parse_version": "raw_only_v1"
    }

### (3-3)폴더전체hwp를 raw_only로 파싱

In [18]:
def parse_hwp_folder_raw(folder_path: str) -> pd.DataFrame:
    """
    폴더 내 모든 .hwp 파일에 대해 raw_text만 추출
    """
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"폴더가 존재하지 않습니다: {folder_path}")

    hwp_files = sorted([f for f in folder.rglob("*.hwp") if f.is_file()])

    print(f"대상 HWP 파일 수: {len(hwp_files)}")

    results = []

    for i, file_path in enumerate(hwp_files, 1):
        print(f"[{i}/{len(hwp_files)}] {file_path.name}")

        row = parse_single_hwp_raw(str(file_path))
        results.append(row)

        print(
            f"  - status={row['status']}, "
            f"raw_len={row['raw_text_len']}, "
            f"time={row['processing_time']}초"
        )

    df = pd.DataFrame(results)
    return df

### (3-4)실행

In [19]:
folder_path = "/home/shared/files"

df_parsed_hwp_v1 = parse_hwp_folder_raw(folder_path=folder_path)

대상 HWP 파일 수: 94
[1/94] (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp
  - status=success, raw_len=107548, time=0.11초
[2/94] (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp
  - status=success, raw_len=53928, time=0.05초
[3/94] (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp
  - status=success, raw_len=76535, time=0.07초
[4/94] (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp
  - status=success, raw_len=44591, time=0.04초
[5/94] 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
  - status=success, raw_len=49069, time=0.04초
[6/94] BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp
  - status=success, raw_len=72609, time=0.06초
[7/94] KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
  - status=success, raw_len=135258, time=0.16초
[8/94] 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  - status=success, raw_len=68037, time=0.05초
[9/94] 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp
  - status=success, raw_len=85795, time=0.06초
[10/94] 경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp
  - status=success, raw_len=72728, time=0.06초
[11/94] 경상북도 봉화군_봉

#### (3-4-1)결과확인

In [20]:
print(df_parsed_hwp_v1.shape)
print(df_parsed_hwp_v1.columns.tolist())
df_parsed_hwp_v1.head(3)

(94, 10)
['파일명', '파일형식', '파일경로', 'raw_text', 'raw_text_len', 'raw_parse_success', 'status', 'parse_warning', 'processing_time', 'parse_version']


,파일명,파일형식,파일경로,raw_text,raw_text_len,raw_parse_success,status,parse_warning,processing_time,parse_version
0,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp,hwp,/home/shared/files/(사)벤처기업협회_2024년 벤처확인종합관리시스템...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n\n氠瑢\n\n\n\n\n20...,107548,True,success,None,0.11,raw_only_v1
1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,hwp,/home/shared/files/(사)부산국제영화제_2024년 BIFF & ACF...,捤獥\n\n\n\n汤捯\n\n\n\n歭扯\n\n\n\n湰灧\n\n\n\n\n氠瑢\n...,53928,True,success,None,0.05,raw_only_v1
2,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,hwp,/home/shared/files/(사）한국대학스포츠협의회_KUSF 체육특기자 경기...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n\n氠瑢\n\n\n\n\nKU...,76535,True,success,None,0.07,raw_only_v1


In [21]:
# status
df_parsed_hwp_v1["status"].value_counts(dropna=False)

status
success    94
Name: count, dtype: int64

In [22]:
# 성공여부
df_parsed_hwp_v1["raw_parse_success"].value_counts(dropna=False)

raw_parse_success
True    94
Name: count, dtype: int64

#### (3-4-2)문제점검

In [23]:
# raw_text가 비어 있는 파일
df_parsed_hwp_v1.loc[
    df_parsed_hwp_v1["raw_text_len"] == 0,
    ["파일명", "status", "parse_warning", "processing_time"]
]

,파일명,status,parse_warning,processing_time


In [24]:
# 텍스트가 짧은 파일 상위20
df_parsed_hwp_v1.sort_values("raw_text_len", ascending=True)[
    ["파일명", "raw_text_len", "status", "parse_warning"]
].head(20)

,파일명,raw_text_len,status,parse_warning
3,(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp,44591,success,None
93,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,44991,success,None
84,한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp,45343,success,None
52,재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp,45831,success,None
63,한국교육과정평가원_국가교육과정정보센터(NCIC) 시스템 운영 및 개선.hwp,45875,success,None
11,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,46976,success,None
55,조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp,47017,success,None
4,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,49069,success,None
78,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,49409,success,None
49,재단법인 한국장애인문화예술원_2024년 장애인문화예술정보시스템 이음.hwp,50524,success,None


### (3-5)저장

In [25]:
save_dir

PosixPath('/home/bidcoin')

In [26]:
save_dir = "/home/bidcoin"  
os.makedirs(save_dir, exist_ok=True)

csv_path = os.path.join(save_dir, "df_parsed_hwp_v1.csv")
df_parsed_hwp_v1.to_csv(csv_path, index=False, encoding="utf-8")

print("CSV 저장 완료:", csv_path)

CSV 저장 완료: /home/bidcoin/df_parsed_hwp_v1.csv


In [27]:
pkl_path = os.path.join(save_dir, "df_parsed_hwp_v1.pkl")
df_parsed_hwp_v1.to_pickle(pkl_path)

print("PKL 저장 완료:", pkl_path)

PKL 저장 완료: /home/bidcoin/df_parsed_hwp_v1.pkl


In [28]:
df_parsed_hwp_v1.shape

(94, 10)

In [29]:
df_parsed_hwp_v1.columns

Index(['파일명', '파일형식', '파일경로', 'raw_text', 'raw_text_len', 'raw_parse_success',
       'status', 'parse_warning', 'processing_time', 'parse_version'],
      dtype='str')

## (4)hwp 정제

### (4-1)정제함수

In [30]:
import re
import pandas as pd
import unicodedata

def clean_text_func(text):
    """
    raw_text 기준 보수적 공통 정제

    원칙
    - 사업명, 예산, 기간, URL, 이메일, 요구사항 코드는 보존
    - 줄 단위 반복 노이즈 + 문장 중간 한자형 노이즈 제거
    - 제목형 띄어쓰기만 제한적으로 복원
    - 숫자/표/항목 구조는 최대한 유지
    """
    if pd.isna(text):
        return text

    text = str(text)
    if not text.strip():
        return text

    # --------------------------------------------------------------------------
    # 패턴 정의
    # --------------------------------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"(?:https?://|www\.)\S+"                                            # URL
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"                   # email
        r"|(?:TEL|FAX|전화|팩스)\s*[:：]?\s*\+?\d{1,4}[-)\s]?\d{2,4}-\d{3,4}" # 연락처
        r"|\b[A-Z]{2,5}\s*-\s*\d{2,4}\b"                                     # CNR-001, PMR-002
        r"|(?:[A-Za-z]:)?[\\/][^\s]+"                                        # 경로 비슷한 값
        r")",
        re.IGNORECASE
    )

    protected_token_pattern = re.compile(
        r"("
        r"^(?:https?://|www\.)\S+$"
        r"|^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        r"|^(?:TEL|FAX|전화|팩스)$"
        r"|^\+?\d{1,4}[-)\s]?\d{2,4}-\d{3,4}$"
        r"|^[A-Z]{2,5}\s*-\s*\d{2,4}$"
        r"|^(?:[A-Za-z]:)?[\\/].+$"
        r")",
        re.IGNORECASE
    )

    noise_line_pattern = re.compile(
        r"^\s*(?:"
        r"[氠瑢汤捯桤灧湯湷湰灧漠杳捤獥]+"
        r"|[Āࢀ]+"
        r"|[汫╨]+"
        r"|(?:[^\w가-힣]{1,5})"
        r")\s*$"
    )

    cjk_noise_token_pattern = re.compile(r"^[\u4E00-\u9FFF]{2,6}$")
    cjk_noise_line_pattern = re.compile(r"^\s*(?:[\u4E00-\u9FFF]{2,6}\s*){1,20}$")

    explicit_noise_token_pattern = re.compile(
        r"^(?:氠瑢|汤捯|桤灧|湯湷|湰灧|漠杳|捤獥|Ā|ࢀ|汫╨)$"
    )

    inline_noise_pattern = re.compile(
        r"(?:氠瑢|汤捯|桤灧|湯湷|湰灧|漠杳|捤獥|Ā|ࢀ|汫╨)"
    )

    spaced_ko_title_pattern = re.compile(
        r"(?<![가-힣A-Za-z0-9])(?:[가-힣]\s){1,20}[가-힣](?![가-힣A-Za-z0-9])"
    )

    # --------------------------------------------------------------------------
    # 0) 유니코드 정규화
    # --------------------------------------------------------------------------
    text = unicodedata.normalize("NFKC", text)

    # 1) 줄바꿈/탭 정리
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\t", " ")

    # 2) 제어문자 제거
    text = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", " ", text)

    # 3) 자주 보이는 특수 깨짐 문자 제거
    text = re.sub(r"[↸ᬄὩ⇟]", " ", text)

    # --------------------------------------------------------------------------
    # 4) 줄 단위 처리
    # --------------------------------------------------------------------------
    cleaned_lines = []

    for raw_line in text.split("\n"):
        line = raw_line.strip()

        if not line:
            cleaned_lines.append("")
            continue

        # 보호 줄은 거의 그대로 둠
        if protected_line_pattern.search(line):
            line = re.sub(r"[ ]{2,}", " ", line).strip()
            cleaned_lines.append(line)
            continue

        # 줄 전체가 노이즈면 제거
        if noise_line_pattern.fullmatch(line):
            continue

        # 줄 전체가 짧은 한자 노이즈 덩어리면 제거
        if cjk_noise_line_pattern.fullmatch(line):
            continue

        # 문장 안 대표 노이즈 제거
        line = inline_noise_pattern.sub(" ", line)

        # 토큰 단위 노이즈 제거
        tokens = line.split()
        kept_tokens = []

        for tok in tokens:
            if protected_token_pattern.search(tok):
                kept_tokens.append(tok)
                continue

            if explicit_noise_token_pattern.fullmatch(tok):
                continue

            if cjk_noise_token_pattern.fullmatch(tok):
                continue

            kept_tokens.append(tok)

        line = " ".join(kept_tokens)

        # 연속 기호 노이즈 축소
        line = re.sub(r"(?:[‧·•∙]{3,})", " ", line)

        # 공백 정리
        line = re.sub(r"[ ]{2,}", " ", line).strip()

        if not line:
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # --------------------------------------------------------------------------
    # 5) 제목형 띄어쓰기 복원
    # --------------------------------------------------------------------------
    def restore_spaced_title(match):
        s = match.group(0)
        compact = s.replace(" ", "")

        if not (2 <= len(compact) <= 25):
            return s
        if re.search(r"(https?|www\.|@|\d{2,4}-\d{2,4}-\d{3,4})", s, re.I):
            return s
        if re.search(r"\b[A-Z]{2,5}\s*-\s*\d{2,4}\b", s):
            return s
        if re.fullmatch(r"(?:[가-힣]\s){1,20}[가-힣]", s.strip()):
            return compact

        return s

    text = spaced_ko_title_pattern.sub(restore_spaced_title, text)

    # --------------------------------------------------------------------------
    # 6) 자주 나오는 표제어 보정
    # --------------------------------------------------------------------------
    replacements = {
        "제안요청서": "제안요청서",
        "목차": "목차",
        "사업명": "사업명",
        "과업명": "과업명",
        "사업기간": "사업기간",
        "사업예산": "사업예산",
        "제안요청내용": "제안요청 내용",
        "제안요청사항": "제안요청 사항",
        "사업개요": "사업개요",
    }

    for k, v in replacements.items():
        text = re.sub(rf"\b{k}\b", v, text)

    # --------------------------------------------------------------------------
    # 7) 목차 줄 끝 페이지 번호 제거
    # --------------------------------------------------------------------------
    final_lines = []
    for line in text.split("\n"):
        if protected_line_pattern.search(line):
            final_lines.append(line)
            continue

        line = re.sub(r"(\s*[-·•‧∙]\s*\d{1,3})$", "", line).rstrip()
        final_lines.append(line)

    text = "\n".join(final_lines)

    # 8) 빈 줄 과다 축소
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

### (4-2)함수적용

In [31]:
df_parsed_hwp_v1["clean_text"] = df_parsed_hwp_v1["raw_text"].apply(clean_text_func)

In [32]:
df_parsed_hwp_v1.columns

Index(['파일명', '파일형식', '파일경로', 'raw_text', 'raw_text_len', 'raw_parse_success',
       'status', 'parse_warning', 'processing_time', 'parse_version',
       'clean_text'],
      dtype='str')

#### (4-2-1)문자수 변화확인

In [33]:
# 문자수 컬럼 생성

df_parsed_hwp_v1["clean_text_len"] = df_parsed_hwp_v1["clean_text"].fillna("").str.len()

# 감소 문자수
df_parsed_hwp_v1["공통정제_초기대비_감소문자수"] = df_parsed_hwp_v1["raw_text_len"] - df_parsed_hwp_v1["clean_text_len"]

# 감소율(%)
df_parsed_hwp_v1["공통정제_초기대비_감소율(%)"] = (
    df_parsed_hwp_v1["공통정제_초기대비_감소문자수"] / df_parsed_hwp_v1["raw_text_len"].replace(0, pd.NA) * 100
)

# 보기 좋게 반올림
rate_cols = [
    "공통정제_초기대비_감소율(%)",
]
df_parsed_hwp_v1[rate_cols] = df_parsed_hwp_v1[rate_cols].round(2)

In [34]:
df_parsed_hwp_v1.sort_values(
    by="공통정제_초기대비_감소율(%)",
    ascending=False
)

,파일명,파일형식,파일경로,raw_text,raw_text_len,raw_parse_success,status,parse_warning,processing_time,parse_version,clean_text,clean_text_len,공통정제_초기대비_감소문자수,공통정제_초기대비_감소율(%)
31,부산관광공사_경영정보시스템 기능개선.hwp,hwp,/home/shared/files/부산관광공사_경영정보시스템 기능개선.hwp,捤獥\n\n\n\n汤捯\n\n\n\n\n湰灧\n\n\n\n湯湷\n\n\n\n氠瑢\n...,80582,True,success,None,0.06,raw_only_v1,제안요청서\n[용역명 : 경영정보시스템 기능개선]\n2024. 03.\n\n목차 -...,71654,8928,11.08
14,광주과학기술원_학사시스템 기능개선 사업.hwp,hwp,/home/shared/files/광주과학기술원_학사시스템 기능개선 사업.hwp,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n漠杳\n\n\n\n漠杳\n\n...,51278,True,success,None,0.04,raw_only_v1,제안요청서\n\n사업명\n학사시스템 기능개선 사업\n주관기관\n광주과학기술원\n\n...,45659,5619,10.96
59,케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp,hwp,/home/shared/files/케빈랩 주식회사_평택시 강소형 스마트시티 AI 기...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n氠瑢\n\n\n\n\n漠杳\n...,50743,True,success,None,0.05,raw_only_v1,평택시 강소형 스마트시티 조성사업 영상 AI감지 및 홍수감시 연동 시스템 구축 사업...,45591,5152,10.15
48,재단법인 광주연구원_광주정책연구아카이브(GPA) 시스템 개발.hwp,hwp,/home/shared/files/재단법인 광주연구원_광주정책연구아카이브(GPA) ...,捤獥\n\n\n\n汤捯\n\n\n\n\n氠瑢\n\n\n\n\n漠杳\n\n\n\n\n...,75735,True,success,None,0.06,raw_only_v1,제안요청서\n\n과업명\n광주정책연구아카이브(GPA) 시스템 개발\n\n2024. ...,68121,7614,10.05
13,광주과학기술원_실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업.hwp,hwp,/home/shared/files/광주과학기술원_실시간통합연구비관리시스템(RCMS)...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n漠杳\n\n\n\n漠杳\n\n...,62487,True,success,None,0.06,raw_only_v1,제안요청서\n\n사업명\n실시간통합연구비관리시스템(RCMS)\n연계 모듈 변경 사업...,56270,6217,9.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,hwp,/home/shared/files/(사）한국대학스포츠협의회_KUSF 체육특기자 경기...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n\n氠瑢\n\n\n\n\nKU...,76535,True,success,None,0.07,raw_only_v1,KUSF 체육특기자 경기기록 관리시스템 개발 제안요청서\n2024. 8.\n\n【목...,73228,3307,4.32
23,그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp,hwp,/home/shared/files/그랜드코리아레저(주)_2024년도 GKL 그룹웨...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n桤灧\n\n\n\n\n \n제...,152619,True,success,None,0.12,raw_only_v1,제안요청서\n\n사업명\nGKL 그룹웨어 시스템 구축사업\n주관기관\n그랜드코리아레...,146053,6566,4.30
55,조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp,hwp,/home/shared/files/조선대학교_(재공고)2024 조선대학교 SW중심대...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n제 안 요 청 서\n氠瑢\n\...,47017,True,success,None,0.04,raw_only_v1,제안요청서\n\n사업명\n2024 조선대학교 SW중심대학 사업관리시스템(WeHub)...,45004,2013,4.28
61,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,hwp,/home/shared/files/한국가스공사_[재공고]차세대 통합정보시스템(ERP...,捤獥\n\n\n\n汤捯\n\n\n\n湰灧\n\n\n\n\n과 업 지 시 서\n- 차...,102774,True,success,None,0.07,raw_only_v1,과업지시서\n- 차세대 통합정보시스템(ERP) 구축 -\n2024. 8.\n\n목차...,99457,3317,3.23


### (4-3)저장

In [35]:
save_dir = "/home/bidcoin"
os.makedirs(save_dir, exist_ok=True)

csv_path = os.path.join(save_dir, "df_parsed_hwp.csv")
df_parsed_hwp_v1.to_csv(csv_path, index=False, encoding="utf-8")

print("CSV 저장 완료:", csv_path)

CSV 저장 완료: /home/bidcoin/df_parsed_hwp.csv


In [36]:
pkl_path = os.path.join(save_dir, "df_parsed_hwp.pkl")
df_parsed_hwp_v1.to_pickle(pkl_path)

print("PKL 저장 완료:", pkl_path)

PKL 저장 완료: /home/bidcoin/df_parsed_hwp.pkl


***

## (5)pdf,hwp concat

### (5-1)정제후 pdf불러오기

In [37]:
# pdf
df_pdf = pd.read_csv("/home/bidcoin/df_parsed_pdf_hard.csv", encoding="utf-8")
df_pdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   파일명         6 non-null      str  
 1   파일형식        6 non-null      str  
 2   raw_text    6 non-null      str  
 3   clean_text  6 non-null      str  
 4   rag_text    6 non-null      str  
dtypes: str(5)
memory usage: 2.3 MB


In [38]:
df_pdf = df_pdf[["파일명", "파일형식", "raw_text", "rag_text"]].copy()
df_pdf = df_pdf.rename(columns={"rag_text": "clean_text"})

In [39]:
df_pdf.columns

Index(['파일명', '파일형식', 'raw_text', 'clean_text'], dtype='str')

### (5-2)정제후 hwp불러오기

In [40]:
# hwp
df_hwp = pd.read_csv("/home/bidcoin/df_parsed_hwp.csv", encoding="utf-8")
df_hwp.info()

<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   파일명                94 non-null     str    
 1   파일형식               94 non-null     str    
 2   파일경로               94 non-null     str    
 3   raw_text           94 non-null     str    
 4   raw_text_len       94 non-null     int64  
 5   raw_parse_success  94 non-null     bool   
 6   status             94 non-null     str    
 7   parse_warning      0 non-null      float64
 8   processing_time    94 non-null     float64
 9   parse_version      94 non-null     str    
 10  clean_text         94 non-null     str    
 11  clean_text_len     94 non-null     int64  
 12  공통정제_초기대비_감소문자수    94 non-null     int64  
 13  공통정제_초기대비_감소율(%)   94 non-null     float64
dtypes: bool(1), float64(3), int64(3), str(7)
memory usage: 28.5 MB


In [41]:
df_hwp = df_hwp[["파일명", "파일형식", "raw_text", "clean_text"]].copy()
df_hwp.columns

Index(['파일명', '파일형식', 'raw_text', 'clean_text'], dtype='str')

### (5-3)concat

In [42]:
df_merge = pd.concat([df_pdf, df_hwp], ignore_index=True)
df_merge.shape

(100, 4)

In [43]:
output_path = "/home/bidcoin/df_parsed.csv"
df_merge.to_csv(output_path, index=False, encoding="utf-8")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/df_parsed.csv
